# 00.run-tests

- Reconciliation and integrity checks across landing, bronze, silver and gold.

What reconciles at each hop, and why it differs:

| Hop | Expected | Reason |
|---|---|---|
| landing to bronze | exactly equal | bronze does no filtering, FAILFAST either passed everything or threw |
| bronze to silver | silver equals distinct non-null grain in bronze | silver dedupes and drops null keys, so it can only shrink, by a knowable amount |
| silver to gold | exactly equal | the dimension joins should not drop facts |

- Lakehouse names, landing path and the flight grain

In [16]:
# One constant to change when you run this against prod.
WORKSPACE = "aeropulse_dev"        # change to aeropulse_prod to run this in prod
ONELAKE   = f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com"

LANDING_PATH = f"{ONELAKE}/aeropulse_landing_lh.Lakehouse/Files"
BRONZE_PATH  = f"{ONELAKE}/aeropulse__bronze__lh.Lakehouse/Tables/dbo"
SILVER_PATH  = f"{ONELAKE}/aeropulse_silver_lh.Lakehouse/Tables/silver"
GOLD_PATH    = f"{ONELAKE}/aeropulse_gold_lh.Lakehouse/Tables/dbo"

# The columns silver deduplicates on. Adjust to match flight-bronze-to-silver.

FLIGHT_GRAIN = ['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM',
                'ORIGIN', 'DEST']

##FLIGHT_GRAIN = ["flight_date", "carrier_code", "flight_number",
                ###"origin_airport_code", "destination_airport_code"]

# Every path must sit in the same workspace, or you are comparing dev against prod
# and the mismatch tells you nothing about your data.
for label, p in [("landing", LANDING_PATH), ("bronze", BRONZE_PATH),
                 ("silver", SILVER_PATH),   ("gold",   GOLD_PATH)]:
    assert f"//{WORKSPACE}@" in p, f"{label} path is not in {WORKSPACE}: {p}"
    print(f"{label:8} {p}")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 18, Finished, Available, Finished, False)

landing  abfss://aeropulse_dev@onelake.dfs.fabric.microsoft.com/aeropulse_landing_lh.Lakehouse/Files
bronze   abfss://aeropulse_dev@onelake.dfs.fabric.microsoft.com/aeropulse__bronze__lh.Lakehouse/Tables/dbo
silver   abfss://aeropulse_dev@onelake.dfs.fabric.microsoft.com/aeropulse_silver_lh.Lakehouse/Tables/silver
gold     abfss://aeropulse_dev@onelake.dfs.fabric.microsoft.com/aeropulse_gold_lh.Lakehouse/Tables/dbo


- Confirms each path points at the table folders you expect. If silver lists a schema
folder rather than table names, add it to SILVER_PATH.

In [17]:
import notebookutils

for label, p in [("landing", LANDING_PATH), ("bronze", BRONZE_PATH),
                 ("silver", SILVER_PATH),   ("gold",   GOLD_PATH)]:
    try:
        entries = sorted(e.name for e in notebookutils.fs.ls(p))
        print(f"{label:8} {', '.join(entries) if entries else '(empty)'}")
    except Exception as e:
        print(f"{label:8} not readable: {e}")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 19, Finished, Available, Finished, False)

landing  airport, carrier, flight
bronze   bronze_airport, bronze_carrier, bronze_flight
silver   airport, carrier, flight, schema.json.gz
gold     dim_carrier, dim_date, dim_destination_airport, dim_origin_airport, fact_flight


- load every layer by path and register temp views

Every check below uses the short view names.

In [18]:
def load(base, table, view):
    df = spark.read.format("delta").load(f"{base}/{table}")
    df.createOrReplaceTempView(view)
    return df

def landing(source):
    """recursiveFileLookup picks up the batch_id subfolders under flight."""
    return (spark.read
            .option("header", True)
            .option("recursiveFileLookup", "true")
            .csv(f"{LANDING_PATH}/{source}"))

load(BRONZE_PATH, "bronze_airport", "bronze_airport")
load(BRONZE_PATH, "bronze_carrier", "bronze_carrier")
load(BRONZE_PATH, "bronze_flight",  "bronze_flight")

load(SILVER_PATH, "airport", "silver_airport")
load(SILVER_PATH, "carrier", "silver_carrier")
load(SILVER_PATH, "flight",  "silver_flight")

load(GOLD_PATH, "dim_carrier", "dim_carrier")
load(GOLD_PATH, "dim_origin_airport", "dim_origin_airport")
load(GOLD_PATH, "dim_destination_airport", "dim_destination_airport")
load(GOLD_PATH, "dim_date", "dim_date")
load(GOLD_PATH, "fact_flight", "fact_flight")

print("views registered")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 20, Finished, Available, Finished, False)

views registered


- Helpers

In [19]:
from pyspark.sql import functions as F

results = []

def check(name, actual, expected, note=""):
    ok = (actual == expected)
    results.append({"check": name, "expected": str(expected),
                    "actual": str(actual),
                    "status": "PASS" if ok else "FAIL", "note": note})
    print(f"{'PASS' if ok else 'FAIL'}  {name:52} expected {expected}, got {actual}")

def zero(name, query, note=""):
    check(name, spark.sql(query).count(), 0, note)

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 21, Finished, Available, Finished, False)

- check landing to bronze

Bronze does no filtering and no dedup, so these must be exactly equal. A gap means
rows were lost on read, which FAILFAST should have thrown on rather than dropped.

In [20]:
for source, view in [("airport", "bronze_airport"),
                     ("carrier", "bronze_carrier"),
                     ("flight",  "bronze_flight")]:
    l = landing(source).count()
    b = spark.table(view).count()
    check(f"{view} row count matches landing", b, l)

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 22, Finished, Available, Finished, False)

PASS  bronze_airport row count matches landing             expected 6818, got 6818
PASS  bronze_carrier row count matches landing             expected 1751, got 1751
PASS  bronze_flight row count matches landing              expected 3611578, got 3611578


- check landing lineage columns survived into bronze

Every landed row carries batch_id, ingested_timestamp and source_path so any row can
be traced back to the run and file that produced it. If they are missing or null,
that lineage is gone.

In [21]:
for view in ["bronze_airport", "bronze_carrier", "bronze_flight"]:
    cols = spark.table(view).columns
    missing = [c for c in ["batch_id", "ingested_timestamp", "source_path"]
               if c not in cols]
    check(f"{view} has lineage columns", len(missing), 0, str(missing))

zero("no null batch_id in bronze_flight", """
    SELECT batch_id FROM bronze_flight WHERE batch_id IS NULL
""")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 23, Finished, Available, Finished, False)

PASS  bronze_airport has lineage columns                   expected 0, got 0
PASS  bronze_carrier has lineage columns                   expected 0, got 0
PASS  bronze_flight has lineage columns                    expected 0, got 0
PASS  no null batch_id in bronze_flight                    expected 0, got 0


- check bronze columns are all string

Deliberate design. Casting is deferred to silver so a source value that will not
convert cleanly is never lost to a cast that fired too early. This check is what
stops that decision quietly eroding.

In [22]:
non_string = [f.name for f in spark.table("bronze_flight").schema.fields
              if f.dataType.simpleString() != "string"]
check("bronze_flight columns are all string", len(non_string), 0, str(non_string))

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 24, Finished, Available, Finished, False)

PASS  bronze_flight columns are all string                 expected 0, got 0


- check bronze to silver

Silver dedupes and drops null keys, so it can only shrink. The amount it shrinks by
is knowable, so this reconciles exactly rather than settling for "smaller looks about
right".

In [23]:
b = spark.sql("""
    SELECT DISTINCT code FROM bronze_airport WHERE code IS NOT NULL
""").count()
check("silver airport matches bronze distinct codes",
      spark.table("silver_airport").count(), b)

b = spark.sql("""
    SELECT DISTINCT code FROM bronze_carrier WHERE code IS NOT NULL
""").count()
check("silver carrier matches bronze distinct codes",
      spark.table("silver_carrier").count(), b)

grain    = ", ".join(FLIGHT_GRAIN)
not_null = " AND ".join(f"{c} IS NOT NULL" for c in FLIGHT_GRAIN)
b = spark.sql(f"""
    SELECT DISTINCT {grain} FROM bronze_flight WHERE {not_null}
""").count()
check("silver flight matches bronze distinct grain",
      spark.table("silver_flight").count(), b,
      "adjust FLIGHT_GRAIN if this fails")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 25, Finished, Available, Finished, False)

PASS  silver airport matches bronze distinct codes         expected 6818, got 6818
PASS  silver carrier matches bronze distinct codes         expected 1751, got 1751
PASS  silver flight matches bronze distinct grain          expected 3611576, got 3611576


- check every batch reached every layer

Catches a batch that ran halfway and stopped. A missing batch downstream is invisible
to a row count, because the remaining batches still add up to something plausible.

In [24]:
bb = {r[0] for r in spark.sql("SELECT DISTINCT batch_id FROM bronze_flight").collect()}
sb = {r[0] for r in spark.sql("SELECT DISTINCT batch_id FROM silver_flight").collect()}
gb = {r[0] for r in spark.sql("SELECT DISTINCT batch_id FROM fact_flight").collect()}

check("every bronze batch reached silver",    len(bb - sb), 0, str(sorted(bb - sb)))
check("every silver batch reached gold",      len(sb - gb), 0, str(sorted(sb - gb)))
check("no unexpected batch appeared in gold", len(gb - sb), 0, str(sorted(gb - sb)))

print(f"\nbatches: bronze {len(bb)}, silver {len(sb)}, gold {len(gb)}")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 26, Finished, Available, Finished, False)

PASS  every bronze batch reached silver                    expected 0, got 0
PASS  every silver batch reached gold                      expected 0, got 0
PASS  no unexpected batch appeared in gold                 expected 0, got 0

batches: bronze 6, silver 6, gold 6


- check if surrogate keys are unique

In [26]:
for view, key in [
    ("dim_carrier",             "carrier_sk"),
    ("dim_origin_airport",      "origin_airport_sk"),
    ("dim_destination_airport", "airport_destination_sk"),
    ("dim_date",                "date_id"),
    ("fact_flight",             "flight_sk"),
]:
    zero(f"{view}.{key} unique", f"""
        SELECT {key} FROM {view} GROUP BY {key} HAVING COUNT(*) > 1
    """)

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 28, Finished, Available, Finished, False)

PASS  dim_carrier.carrier_sk unique                        expected 0, got 0
PASS  dim_origin_airport.origin_airport_sk unique          expected 0, got 0
PASS  dim_destination_airport.airport_destination_sk unique expected 0, got 0
PASS  dim_date.date_id unique                              expected 0, got 0
PASS  fact_flight.flight_sk unique                         expected 0, got 0


- check silver natural keys and grain

In [27]:
zero("silver airport_code unique", """
    SELECT airport_code FROM silver_airport
    GROUP BY airport_code HAVING COUNT(*) > 1
""")

zero("silver carrier_code unique", """
    SELECT carrier_code FROM silver_carrier
    GROUP BY carrier_code HAVING COUNT(*) > 1
""")

zero("silver flight_sk unique", """
    SELECT flight_sk FROM silver_flight
    GROUP BY flight_sk HAVING COUNT(*) > 1
""")

zero("no null natural keys in silver airport", """
    SELECT airport_code FROM silver_airport WHERE airport_code IS NULL
""")

zero("no null natural keys in silver carrier", """
    SELECT carrier_code FROM silver_carrier WHERE carrier_code IS NULL
""")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 29, Finished, Available, Finished, False)

PASS  silver airport_code unique                           expected 0, got 0
PASS  silver carrier_code unique                           expected 0, got 0
PASS  silver flight_sk unique                              expected 0, got 0
PASS  no null natural keys in silver airport               expected 0, got 0
PASS  no null natural keys in silver carrier               expected 0, got 0


- check gold dimensions are not empty

A full refresh dimension filtered on batch_id returns an empty set rather than an
error, so nothing upstream complains. This is the check that catches it.

In [28]:
for view in ["dim_carrier", "dim_origin_airport",
             "dim_destination_airport", "dim_date"]:
    n = spark.table(view).count()
    check(f"{view} is populated", n > 0, True, f"{n} rows")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 30, Finished, Available, Finished, False)

PASS  dim_carrier is populated                             expected True, got True
PASS  dim_origin_airport is populated                      expected True, got True
PASS  dim_destination_airport is populated                 expected True, got True
PASS  dim_date is populated                                expected True, got True


- check silver to gold

In [29]:
check("fact_flight row count matches silver",
      spark.table("fact_flight").count(), spark.table("silver_flight").count())

check("dim_carrier row count matches silver",
      spark.table("dim_carrier").count(), spark.table("silver_carrier").count())

check("dim_origin_airport row count matches silver",
      spark.table("dim_origin_airport").count(), spark.table("silver_airport").count())

# dim_destination_airport is built from distinct destination codes in silver flight,
# not from silver airport, so it reconciles against that instead.
s = spark.sql("""
    SELECT DISTINCT destination_airport_code FROM silver_flight
    WHERE destination_airport_code IS NOT NULL
""").count()
check("dim_destination_airport matches silver distinct codes",
      spark.table("dim_destination_airport").count(), s)

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 31, Finished, Available, Finished, False)

PASS  fact_flight row count matches silver                 expected 3611576, got 3611576
PASS  dim_carrier row count matches silver                 expected 1751, got 1751
PASS  dim_origin_airport row count matches silver          expected 6818, got 6818
PASS  dim_destination_airport matches silver distinct codes expected 354, got 354


- check referential integrity

In [31]:
zero("no null surrogate keys in fact_flight", """
    SELECT flight_sk FROM fact_flight
    WHERE carrier_sk IS NULL
       OR origin_airport_sk IS NULL
       OR airport_destination_sk IS NULL
       OR flight_date_id IS NULL
""")

zero("fact_flight -> dim_carrier", """
    SELECT f.flight_sk FROM fact_flight f
    LEFT JOIN dim_carrier d ON f.carrier_sk = d.carrier_sk
    WHERE f.carrier_sk IS NOT NULL AND d.carrier_sk IS NULL
""")

zero("fact_flight -> dim_origin_airport", """
    SELECT f.flight_sk FROM fact_flight f
    LEFT JOIN dim_origin_airport d ON f.origin_airport_sk = d.origin_airport_sk
    WHERE f.origin_airport_sk IS NOT NULL AND d.origin_airport_sk IS NULL
""")

zero("fact_flight -> dim_destination_airport", """
    SELECT f.flight_sk FROM fact_flight f
    LEFT JOIN dim_destination_airport d
      ON f.airport_destination_sk = d.airport_destination_sk
    WHERE f.airport_destination_sk IS NOT NULL AND d.airport_destination_sk IS NULL
""")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 33, Finished, Available, Finished, False)

PASS  no null surrogate keys in fact_flight                expected 0, got 0
PASS  fact_flight -> dim_carrier                           expected 0, got 0
PASS  fact_flight -> dim_origin_airport                    expected 0, got 0
PASS  fact_flight -> dim_destination_airport               expected 0, got 0


- resolving date keys

In [32]:
zero("fact_flight.flight_date_id -> dim_date", """
    SELECT f.flight_date_id FROM fact_flight f
    LEFT JOIN dim_date d ON f.flight_date_id = d.date_id
    WHERE f.flight_date_id IS NOT NULL AND d.date_id IS NULL
""")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 34, Finished, Available, Finished, False)

PASS  fact_flight.flight_date_id -> dim_date               expected 0, got 0


- check derived business rules still hold

In [35]:
zero("cancellation flag and code agree", """
    SELECT flight_sk FROM fact_flight
    WHERE (is_cancelled = true
           AND (cancellation_code IS NULL OR cancellation_code = ''))
       OR (is_cancelled = false
           AND cancellation_code IS NOT NULL AND cancellation_code <> '')
""")

zero("is_delayed agrees with the 15 minute rule", """
    SELECT flight_sk FROM fact_flight
    WHERE is_cancelled = false
      AND arrival_delay_minutes IS NOT NULL
      AND is_delayed <> (arrival_delay_minutes >= 15)
""")

zero("no negative total_delay_minutes", """
    SELECT flight_sk FROM fact_flight
    WHERE total_delay_minutes < 0
""")

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 37, Finished, Available, Finished, False)

PASS  cancellation flag and code agree                     expected 0, got 0
PASS  is_delayed agrees with the 15 minute rule            expected 0, got 0
PASS  no negative total_delay_minutes                      expected 0, got 0


# results and raise

In [36]:
display(spark.createDataFrame(results))

failed = [r for r in results if r["status"] == "FAIL"]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")

if failed:
    raise Exception(f"{len(failed)} check(s) failed: "
                    + ", ".join(r["check"] for r in failed))

StatementMeta(, e5dbe9d7-8b96-4537-a813-ec74e3c76315, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c43f15e-d124-437e-90cb-312cf915ddf1)


44 passed, 0 failed
